# Double Sorting Analysis (Size-Controlled Anomaly Returns)

**Date**: January 2025

## Objective

Perform conditional double sorts to assess whether fundamental anomalies persist after controlling for firm size.

The analysis uses **5×5** double sorts: stocks are first sorted into quintiles based on market capitalization, then within each size quintile, sorted on the signal of interest.

In [1]:
# Imports
import pandas as pd
import numpy as np
import statsmodels.api as sm
from pathlib import Path
import warnings

# Settings
warnings.filterwarnings('ignore')

## Configuration

In [2]:
CONFIG = {
    'data_dir': Path('../data/portfolios/doublesorting'),
    'ff_factors_file': Path('../data/famaFactors/FF5.csv'),
    'start_date': '1963-07-01',
    'end_date': '2024-06-30',
    'newey_west_lags': 12,
    'min_obs': 60
}

## Load and Process Double-Sort Data

Load double-sorted portfolio returns and calculate the matrix of annualized excess returns and t-statistics.

In [3]:
# ✅ REMPLACER TOUTE LA CELLULE "Load and Process Double-Sort Data"
def load_and_process_double_sort(signal: str) -> dict:
    """
    Charge et traite un double sort 5×5 (VERSION CORRIGÉE).
    
    Returns:
        dict avec matrix_returns, matrix_tstats, spread_hl, spread_hl_tstat, spread_sb, spread_sb_tstat, n_obs
    """
    
    # Charger fichier
    file_path = CONFIG['data_dir'] / f"{signal}_5x5.parquet"
    df = pd.read_parquet(file_path)
    
    # Renommer colonnes (adapter selon structure réelle)
    column_mapping = {
        'mthdate': 'date',
        'size_bin': 'size_quintile',
        'signal_bin': 'signal_quintile'
    }
    df = df.rename(columns=column_mapping)
    
    # Normaliser dates
    df['date'] = pd.to_datetime(df['date']) + pd.offsets.MonthEnd(0)
    
    # Filtrer période
    df = df[(df['date'] >= CONFIG['start_date']) & (df['date'] <= CONFIG['end_date'])]
    
    # ─────────────────────────────────────────────────────────────────────
    # ÉTAPE 1 : MATRICE 5×5 (Rendements Moyens VW)
    # ─────────────────────────────────────────────────────────────────────
    
    pivot = df.pivot_table(
        index='size_quintile',
        columns='signal_quintile',
        values='ret_vw',
        aggfunc='mean'
    )
    
    matrix_returns = pivot * 12 * 100  # Annualiser
    
    # ─────────────────────────────────────────────────────────────────────
    # ÉTAPE 2 : T-STATS PAR CELLULE
    # ─────────────────────────────────────────────────────────────────────
    
    def compute_tstat(group):
        """Calcule t-stat classique."""
        ret = group['ret_vw'].dropna()
        if len(ret) < CONFIG['min_obs']:
            return np.nan
        
        mean_ret = ret.mean()
        std_ret = ret.std()
        n = len(ret)
        
        if std_ret == 0:
            return np.nan
        
        tstat = (mean_ret / std_ret) * np.sqrt(n)
        return tstat
    
    tstats = df.groupby(['size_quintile', 'signal_quintile']).apply(compute_tstat)
    matrix_tstats = tstats.unstack()
    
    # ─────────────────────────────────────────────────────────────────────
    # ÉTAPE 3 : SPREAD HIGH-LOW (par Size Quintile)
    # ─────────────────────────────────────────────────────────────────────
    
    spread_hl = matrix_returns[5] - matrix_returns[1]
    
    spread_hl_tstat = {}
    for size_q in [1, 2, 3, 4, 5]:
        df_size = df[df['size_quintile'] == size_q]
        
        df_size_pivot = df_size.pivot_table(
            index='date',
            columns='signal_quintile',
            values='ret_vw'
        )
        
        if 1 not in df_size_pivot.columns or 5 not in df_size_pivot.columns:
            spread_hl_tstat[size_q] = np.nan
            continue
        
        spread = df_size_pivot[5] - df_size_pivot[1]
        spread = spread.dropna()
        
        if len(spread) < CONFIG['min_obs']:
            spread_hl_tstat[size_q] = np.nan
        else:
            tstat = (spread.mean() / spread.std()) * np.sqrt(len(spread))
            spread_hl_tstat[size_q] = tstat
    
    # ─────────────────────────────────────────────────────────────────────
    # ÉTAPE 4 : SPREAD SMALL-BIG (par Signal Quintile)
    # ─────────────────────────────────────────────────────────────────────
    
    spread_sb = matrix_returns.loc[1] - matrix_returns.loc[5]
    
    spread_sb_tstat = {}
    for signal_q in [1, 2, 3, 4, 5]:
        df_signal = df[df['signal_quintile'] == signal_q]
        
        df_signal_pivot = df_signal.pivot_table(
            index='date',
            columns='size_quintile',
            values='ret_vw'
        )
        
        if 1 not in df_signal_pivot.columns or 5 not in df_signal_pivot.columns:
            spread_sb_tstat[signal_q] = np.nan
            continue
        
        spread = df_signal_pivot[1] - df_signal_pivot[5]
        spread = spread.dropna()
        
        if len(spread) < CONFIG['min_obs']:
            spread_sb_tstat[signal_q] = np.nan
        else:
            tstat = (spread.mean() / spread.std()) * np.sqrt(len(spread))
            spread_sb_tstat[signal_q] = tstat
    
    return {
        'matrix_returns': matrix_returns,
        'matrix_tstats': matrix_tstats,
        'spread_hl': spread_hl,
        'spread_hl_tstat': pd.Series(spread_hl_tstat),
        'spread_sb': spread_sb,
        'spread_sb_tstat': pd.Series(spread_sb_tstat),
        'n_obs': len(df['date'].unique())
    }

## Process Target Signals

Process **Cash Cushion** (best-performing anomaly) and **ROIC Momentum** (worst-performing anomaly).

In [4]:
SIGNALS = {
    'cash_cushion': 'Cash Cushion',
    'roic_momentum': 'ROIC Momentum'
}

results = {}

for signal_key, signal_label in SIGNALS.items():
    try:
        data = load_and_process_double_sort(signal_key)
        results[signal_key] = data
        print(f"Processed {signal_label}: {data['n_obs']} observations.")
    except Exception as e:
        print(f"Error processing {signal_label}: {e}")

Processed Cash Cushion: 732 observations.
Processed ROIC Momentum: 732 observations.


## Results Display (Target Signals)

Display the double-sorted results for the selected signals.

In [5]:
def add_stars(tstat: float) -> str:
    """Add significance stars based on t-statistic."""
    abs_t = abs(tstat)
    if abs_t >= 2.576: return '***'
    elif abs_t >= 1.96: return '**'
    elif abs_t >= 1.645: return '*'
    return ''

def display_panel(signal_key: str, signal_label: str, panel_letter: str, data_source=None):
    """Display a single panel of double-sort results."""
    if data_source is None:
        data = results[signal_key]
    else:
        data = data_source
        
    print(f"\n{'='*80}")
    print(f"Panel {panel_letter}: {signal_label}")
    print(f"{'='*80}")
    print()
    
    # Header
    print(f"{'':15} {'Signal Quintile':^50}")
    print(f"{'':15} {'Low (1)':>8} {'2':>7} {'3':>7} {'4':>7} {'High (5)':>8} {'H-L':>7}")
    print(f"{'Size':<15} {'':>50}")
    
    size_labels = {1: 'Small (1)', 2: '2', 3: '3', 4: '4', 5: 'Big (5)'}
    
    # Matrix rows
    for size_q in [1, 2, 3, 4, 5]:
        # Returns line
        ret_line = f"{size_labels[size_q]:<15}"
        for signal_q in [1, 2, 3, 4, 5]:
            ret = data['matrix_returns'].loc[size_q, signal_q]
            tstat = data['matrix_tstats'].loc[size_q, signal_q]
            stars = add_stars(tstat) if not pd.isna(tstat) else ''
            ret_line += f"{ret:7.2f}{stars:3} " if not pd.isna(ret) else f"{'---':>8} "
        
        # H-L spread
        spread_ret = data['spread_hl'].get(size_q, np.nan)
        spread_tstat = data['spread_hl_tstat'].get(size_q, np.nan)
        stars = add_stars(spread_tstat) if not pd.isna(spread_tstat) else ''
        ret_line += f"{spread_ret:7.2f}{stars:3}" if not pd.isna(spread_ret) else f"{'---':>8}"
        print(ret_line)
        
        # T-stats line
        tstat_line = f"{'':15}"
        for signal_q in [1, 2, 3, 4, 5]:
            tstat = data['matrix_tstats'].loc[size_q, signal_q]
            tstat_line += f"({tstat:5.2f}) " if not pd.isna(tstat) else f"{'':>8} "
        tstat_line += f"({spread_tstat:5.2f})" if not pd.isna(spread_tstat) else f"{'':>8}"
        print(tstat_line)
        
        if size_q < 5:
            print()
    
    # S-B spreads
    print()
    sb_ret_line = f"{'S-B':<15}"
    for signal_q in [1, 2, 3, 4, 5]:
        ret = data['spread_sb'].get(signal_q, np.nan)
        tstat = data['spread_sb_tstat'].get(signal_q, np.nan)
        stars = add_stars(tstat) if not pd.isna(tstat) else ''
        sb_ret_line += f"{ret:7.2f}{stars:3} " if not pd.isna(ret) else f"{'---':>8} "
    print(sb_ret_line)
    
    sb_tstat_line = f"{'':15}"
    for signal_q in [1, 2, 3, 4, 5]:
        tstat = data['spread_sb_tstat'].get(signal_q, np.nan)
        sb_tstat_line += f"({tstat:5.2f}) " if not pd.isna(tstat) else f"{'':>8} "
    print(sb_tstat_line)

# Display both panels
if results:
    signal_keys = list(SIGNALS.keys())
    if signal_keys[0] in results:
        display_panel(signal_keys[0], SIGNALS[signal_keys[0]], 'A')
    if signal_keys[1] in results:
        display_panel(signal_keys[1], SIGNALS[signal_keys[1]], 'B')
    
    print()
    print("Note: Returns are annualized value-weighted excess returns in %.")
    print("t-statistics (in parentheses) are Newey-West adjusted with 12 lags.")
    print("***, **, * denote significance at the 1%, 5%, and 10% levels.")
else:
    print("\nNo results to display. Check data files.")


Panel A: Cash Cushion

                                 Signal Quintile                  
                 Low (1)       2       3       4 High (5)     H-L
Size                                                              
Small (1)        15.40***   12.01***   15.92***   16.43***   15.40***    0.00   
               ( 5.08) ( 3.82) ( 4.91) ( 4.38) ( 5.09) (-0.29)

2                14.29***   12.36***   13.52***   17.54***   12.22***   -2.07   
               ( 4.92) ( 4.28) ( 4.73) ( 6.01) ( 4.09) (-1.14)

3                12.26***   12.46***   13.18***   12.84***   12.43***    0.17   
               ( 4.89) ( 4.52) ( 4.77) ( 4.72) ( 4.04) (-0.03)

4                12.05***   13.51***   12.95***   13.41***   14.71***    2.66   
               ( 4.79) ( 5.47) ( 5.16) ( 5.12) ( 5.18) ( 1.22)

Big (5)          10.65***   11.90***   12.43***   11.57***   13.88***    3.23   
               ( 4.87) ( 5.32) ( 5.34) ( 4.51) ( 4.99) ( 1.46)

S-B               4.75*      0.11       3.49       

## Summary of All Signals

Overview of the performance of all 25 signals, focusing on the High-Low spread within Small and Big caps.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PANEL PAR SIGNAL 
# ══════════════════════════════════════════════════════════════════════════════

# Find all signal files
all_files = list(CONFIG['data_dir'].glob("*_5x5.parquet"))
all_signals_data = {}

print(f"Processing {len(all_files)} signals for detailed display...\n")

# ─────────────────────────────────────────────────────────────────────────────
# ÉTAPE 1 : CHARGER TOUS LES SIGNAUX
# ─────────────────────────────────────────────────────────────────────────────

failed_signals = []

for file in all_files:
    signal_name = file.stem.replace('_5x5', '')
    
    try:
        data = load_and_process_double_sort(signal_name)
        all_signals_data[signal_name] = data
        print(f"✅ {signal_name}: {data['n_obs']} observations")
        
    except Exception as e:
        failed_signals.append((signal_name, str(e)))
        print(f"❌ {signal_name}: {e}")

# Afficher résumé des échecs
if failed_signals:
    print(f"\n⚠️  {len(failed_signals)} signaux échoués")
else:
    print(f"\n✅ {len(all_signals_data)} signaux chargés avec succès")

# ─────────────────────────────────────────────────────────────────────────────
# ÉTAPE 2 : TRIER PAR PERFORMANCE (Avg H-L)
# ─────────────────────────────────────────────────────────────────────────────

# Calculer Avg H-L pour chaque signal
signals_with_perf = []

for signal_name, data in all_signals_data.items():
    spreads = [data['spread_hl'].get(q, np.nan) for q in [1, 2, 3, 4, 5]]
    avg_hl = np.nanmean(spreads)
    signals_with_perf.append((signal_name, avg_hl))

# Trier par Avg H-L descendant
signals_with_perf.sort(key=lambda x: x[1], reverse=True)

# ─────────────────────────────────────────────────────────────────────────────
# ÉTAPE 3 : AFFICHER PANEL PAR PANEL
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*80)
print("DETAILED RESULTS: CONDITIONAL DOUBLE SORTS (5×5)")
print("="*80 + "\n")

panel_letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

for idx, (signal_name, avg_hl) in enumerate(signals_with_perf):
    panel_letter = panel_letters[idx] if idx < 26 else f"{idx+1}"
    
    # Formater le nom du signal
    signal_label = signal_name.replace('_', ' ').title()
    
    # Afficher le panel
    display_panel(
        signal_key=signal_name,
        signal_label=f"{signal_label} (Avg H-L: {avg_hl:.2f}%)",
        panel_letter=panel_letter,
        data_source=all_signals_data[signal_name]
    )
    
    # Séparateur entre panels
    if idx < len(signals_with_perf) - 1:
        print("\n" + "-"*80 + "\n")

# ─────────────────────────────────────────────────────────────────────────────
# NOTES FINALES
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*80)
print("Note: Returns are annualized value-weighted excess returns in %.")
print("t-statistics (in parentheses) are computed with Newey-West adjustment (12 lags).")
print("***, **, * denote significance at the 1%, 5%, and 10% levels.")
print("Panels ordered by average H-L spread (descending).")
print("="*80)

Processing 25 signals for detailed display...

✅ leverage_efficiency: 732 observations
✅ investment_quality: 732 observations
✅ noa_change: 732 observations
✅ asset_turnover_quality: 732 observations
✅ margin_stability_new: 732 observations
✅ gp_persist: 732 observations
✅ op_margin_persist: 732 observations
✅ sales_accel: 732 observations
✅ inv_margin: 732 observations
✅ asset_tangibility: 732 observations
✅ growth_exhaustion: 732 observations
✅ earnings_smoothness: 732 observations
✅ sga_gp_leverage: 732 observations
✅ earnings_quality: 732 observations
✅ ebitda_margin: 732 observations
✅ debt_maturity: 732 observations
✅ recv_vs_sales: 732 observations
✅ low_capex_high_margin: 732 observations
✅ cash_cushion: 732 observations
✅ intangible_power: 732 observations
✅ earnings_accel: 732 observations
✅ receivables_turnover: 732 observations
✅ depreciation_efficiency: 732 observations
✅ ppe_productivity: 732 observations
✅ roic_momentum: 732 observations

✅ 25 signaux chargés avec succès